In [1]:
import torch
import transformers

from datasets import Dataset, load_dataset
from enum import Enum


from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor

from pprint import pprint
from pydantic import BaseModel

from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

from tqdm import tqdm
from sklearn.metrics import accuracy_score

import pandas as pd
from tabulate import tabulate

import json

print(transformers.__version__)
print(torch.__version__)

INFO 04-12 09:33:48 [__init__.py:239] Automatically detected platform cuda.
4.51.2
2.6.0+cu124


---

В этой задаче вам предложено провести расширенное сравнительное исследование, оценив эффективность нескольких LLM в различных форматах: zero-/few-shot.

---
**Форматы zero-/few-shot**

Рассмотрим два довольно популярных подхода.
* *Zero-Shot* 
* *Few-Shot*

Обратим внимание, что Few-Shot-подход можно реализовать разными способами в зависимости от API или чат-интерфейса модели.

---

Итак, вам требуется применить **две** open-source LLM для задачи sentiment analysis с семинара и сравнить их производительность в режимах zero- и few-shot на **валидационной выборке**. Для few-shot используйте $k=5$ примеров, которые возьмите случайным образом из обучающей выборки. Это факультативная часть курса, поэтому выбор конкретного формата, из описанных выше, остается на вашей стороне :)


В качестве LLM используйте `Qwen/Qwen2.5-3B` и `unsloth/Llama-3.2-1B-Instruct` с HuggingFace. Попробуйте **поэкспериментировать с различными вариантами промпта**.

In [2]:
data = load_dataset("scikit-learn/imdb", split="train").train_test_split(test_size=0.2, seed=42)
data

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 10000
    })
})

Первая модель Zero-Shot Qwen.

In [ ]:
# vLLM поддерживает модели прямо с HF
model = LLM(model="Qwen/Qwen2.5-3B-Instruct", dtype="float16")

In [3]:
# Определяем структуру ответа через Pydantic
class SentimentType(str, Enum):
    positive = "positive"
    negative = "negative"


class ReviewDescription(BaseModel):
    sentiment: SentimentType  # Поле с жестко заданными вариантами

# Автоматически генерируем JSON-схему для валидации:
json_schema = ReviewDescription.model_json_schema()

In [4]:
guided_decoding_params = GuidedDecodingParams(json=json_schema)
sampling_params = SamplingParams(guided_decoding=guided_decoding_params, temperature=0.0)

Я пыталась оценивать по всей валидации, но это получается очень долго. Поэтому я оцениваю по случайному куску из 500 элементов.

In [8]:
subset = data["test"].shuffle(seed=42).select(range(500))

In [9]:
label_map = {"positive": 1, "negative": 0}

true_labels = []
pred_labels = []

for sample in tqdm(subset):
    review = sample["review"]
    true_label = label_map[sample["sentiment"]]

    messages = [{"role": "user", "content": f"Classify this sentiment: {review}"}]
    try:
        outputs = model.chat(messages=messages, sampling_params=sampling_params)
        text = outputs[0].outputs[0].text
        label = label_map[json.loads(text)["sentiment"]]
        pred_labels.append(label)
        true_labels.append(true_label)
    except Exception as e:
        print(f"Ошибка на примере: {e}")


  0%|          | 0/500 [00:00<?, ?it/s]

INFO 04-12 08:13:39 [chat_utils.py:396] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
WARNING 04-12 08:13:39 [__init__.py:33] xgrammar does not support advanced JSON schema features like enums, patterns or numeric ranges. Falling back to use outlines instead.



100%|██████████| 500/500 [12:41<00:00,  1.52s/it]


In [10]:
accuracy_1 = accuracy_score(true_labels, pred_labels)
print(f"Accuracy: {accuracy_1:.4f}")

Accuracy: 0.9240


Теперь реализую Few-Shot

In [11]:
data["train"][:10]

{'review': ["The film disappointed me for many reasons: first of all the depiction of a future which seemed at first realistic to me was well-built but did only feature a marginal role. Then, the story itself was a weak copy of Lost in Translation. The Middle-Eastern setting, man with family meets new girl overseas, karaoke bar, the camera movements and the imagery - all that was a very bad imitation of the excellent Lost in Translation which had also credibility. This movie tries to be something brilliant and cultural: it is not. I wonder why Tim Robbins even considered doing this film!? The female main actress is awful - did she play the precog in Minority Report? And why do you have to show the vagina in a movie like this? Lost in Translation didn't have to show excessive love scenes. R-Rated just for this? This movie isn't even worth watching it from a videostore!",
  'Can this "film" be considered as a film? Imagine the situation: somebody puts a handy cam over a tripod and in fro

In [12]:
subset = data["test"].shuffle(seed=42).select(range(500))

In [ ]:
true_labels = []
pred_labels = []

for sample in tqdm(subset):
    review = sample["review"]
    true_label = label_map[sample["sentiment"]]

    messages = [
        {"role": "user", "content": "You need to classify sentiments. Return only positive or negative. Nothing else."},
        {"role": "system", "content": "ok"},
        {"role": "user", "content": "The film disappointed me for many reasons: first of all the depiction of a future which seemed at first realistic to me was well-built but did only feature a marginal role. Then, the story itself was a weak copy of Lost in Translation. The Middle-Eastern setting, man with family meets new girl overseas, karaoke bar, the camera movements and the imagery - all that was a very bad imitation of the excellent Lost in Translation which had also credibility. This movie tries to be something brilliant and cultural: it is not. I wonder why Tim Robbins even considered doing this film!? The female main actress is awful - did she play the precog in Minority Report? And why do you have to show the vagina in a movie like this? Lost in Translation didn't have to show excessive love scenes. R-Rated just for this? This movie isn't even worth watching it from a videostore!"},
        {"role": "system", "content": "negative"},
        {"role": "user", "content": "Wow, I forgot how great this movie was until I stumbled upon it while looking through the garage. It\'s a kind of strange combination of a bio of Michael Jackson, a collection of musical vignettes, and a story about a super hero fighting to save some little kids. The vignettes are good (especially Speed Demon), but the best part of this movie is the super hero segment, in which Michael Jackson turns into a car, a robot, and finally a spaceship (and it\'s just as weird as it sounds). Joe Pesci is hilarious, and has enough cool imagery and great music to entertain throughout!<br /><br />The real gem however is the incredible Smooth Criminal video, which makes the movie worth owning for that part alone!"},
        {"role": "system", "content": "positive"},
        {"role": "user", "content": "Not really sure what to make of this movie. very weird, very artsy. not the kind of movie you watch because it has a compelling plot or characters. more like the kind of movie that you can't stop watching because of the horrifically fascinating things happening on screen. although, the first time my wife watched this she couldn't make it all the way through... too disturbing for her. runs a bit long, but nonetheless a worthwhile viewing for those interested in very dark movies"},
        {"role": "system", "content": "positive"},
        {"role": "user", "content": "Can this film be considered as a film? Imagine the situation: somebody puts a handy cam over a tripod and in front of a sea promenade and film people walking or jogging along it. Then, he places the camera in a beach, buys some ducks in a pet shop, open their cages and let them run in front of the camera. Later, he just films the water surface and the sound of birds and insects in an absolute darkness. Is it an experiment or just an insult to the audience intelligence? What would it happen if any unknown director did a film like that? Would we mark his job with 10? I always disappoint directors who believe that can do everything they want once they became famous."},
        {"role": "system", "content": "negative"},
        {"role": "user", "content": "Produced by International Playhouse Pictures, it looks as if filmed in a doll house. Everybody's a liar, everything is dream-like, toy-like for no good reason. I'm not saying everything in all movies should be totally realistic, but such unbelievable fantasy things and situations in one movie are way too much. How did they get these fine actors -actresses particularly- to this movie? It's nice to see Mia again; if we were meant to understand why her husband wants to kill her, Mia does do it well. Not funny, not moving, just fake. Stephen Dorff briefly appears at the end, fitting for a play maybe, less for a movie, but this isn't one to measure things at. Terrible."},
        {"role": "system", "content": "negative"},
        {"role": "user", "content": review},
    ]
    try:
        outputs = model.chat(messages=messages, sampling_params=sampling_params)
        text = outputs[0].outputs[0].text
        label = label_map[json.loads(text)["sentiment"]]
        pred_labels.append(label)
        true_labels.append(true_label)
    except Exception as e:
        print(f"Ошибка на примере: {e}")


100%|██████████| 500/500 [14:20<00:00,  1.72s/it]


In [14]:
accuracy_2 = accuracy_score(true_labels, pred_labels)
print(f"Accuracy: {accuracy_2:.4f}")

Accuracy: 0.9400


Аналогично для другой модели.

In [ ]:
model = LLM(model="unsloth/Llama-3.2-1B-Instruct", dtype="float16")

In [7]:
subset = data["test"].shuffle(seed=42).select(range(500))

In [8]:
label_map = {"positive": 1, "negative": 0}
true_labels = []
pred_labels = []

for sample in tqdm(subset):
    review = sample["review"]
    true_label = label_map[sample["sentiment"]]

    messages = [{"role": "user", "content": f"Classify this sentiment: {review}"}]
    try:
        outputs = model.chat(messages=messages, sampling_params=sampling_params)
        text = outputs[0].outputs[0].text
        label = label_map[json.loads(text)["sentiment"]]
        pred_labels.append(label)
        true_labels.append(true_label)
    except Exception as e:
        print(f"Ошибка на примере: {e}")


  0%|          | 0/500 [00:00<?, ?it/s]

INFO 04-12 09:36:12 [chat_utils.py:396] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
WARNING 04-12 09:36:12 [__init__.py:33] xgrammar does not support advanced JSON schema features like enums, patterns or numeric ranges. Falling back to use outlines instead.



100%|██████████| 500/500 [09:27<00:00,  1.13s/it]


In [9]:
accuracy_3 = accuracy_score(true_labels, pred_labels)
print(f"Accuracy: {accuracy_3 :.4f}")

Accuracy: 0.9120


In [10]:
subset = data["test"].shuffle(seed=42).select(range(500))

In [11]:
true_labels = []
pred_labels = []

for sample in tqdm(subset):
    review = sample["review"]
    true_label = label_map[sample["sentiment"]]

    messages = [
        {"role": "user", "content": "You need to classify sentiments. Return only positive or negative. Nothing else."},
        {"role": "system", "content": "ok"},
        {"role": "user", "content": "The film disappointed me for many reasons: first of all the depiction of a future which seemed at first realistic to me was well-built but did only feature a marginal role. Then, the story itself was a weak copy of Lost in Translation. The Middle-Eastern setting, man with family meets new girl overseas, karaoke bar, the camera movements and the imagery - all that was a very bad imitation of the excellent Lost in Translation which had also credibility. This movie tries to be something brilliant and cultural: it is not. I wonder why Tim Robbins even considered doing this film!? The female main actress is awful - did she play the precog in Minority Report? And why do you have to show the vagina in a movie like this? Lost in Translation didn't have to show excessive love scenes. R-Rated just for this? This movie isn't even worth watching it from a videostore!"},
        {"role": "system", "content": "negative"},
        {"role": "user", "content": "Wow, I forgot how great this movie was until I stumbled upon it while looking through the garage. It\'s a kind of strange combination of a bio of Michael Jackson, a collection of musical vignettes, and a story about a super hero fighting to save some little kids. The vignettes are good (especially Speed Demon), but the best part of this movie is the super hero segment, in which Michael Jackson turns into a car, a robot, and finally a spaceship (and it\'s just as weird as it sounds). Joe Pesci is hilarious, and has enough cool imagery and great music to entertain throughout!<br /><br />The real gem however is the incredible Smooth Criminal video, which makes the movie worth owning for that part alone!"},
        {"role": "system", "content": "positive"},
        {"role": "user", "content": "Not really sure what to make of this movie. very weird, very artsy. not the kind of movie you watch because it has a compelling plot or characters. more like the kind of movie that you can't stop watching because of the horrifically fascinating things happening on screen. although, the first time my wife watched this she couldn't make it all the way through... too disturbing for her. runs a bit long, but nonetheless a worthwhile viewing for those interested in very dark movies"},
        {"role": "system", "content": "positive"},
        {"role": "user", "content": "Can this film be considered as a film? Imagine the situation: somebody puts a handy cam over a tripod and in front of a sea promenade and film people walking or jogging along it. Then, he places the camera in a beach, buys some ducks in a pet shop, open their cages and let them run in front of the camera. Later, he just films the water surface and the sound of birds and insects in an absolute darkness. Is it an experiment or just an insult to the audience intelligence? What would it happen if any unknown director did a film like that? Would we mark his job with 10? I always disappoint directors who believe that can do everything they want once they became famous."},
        {"role": "system", "content": "negative"},
        {"role": "user", "content": "Produced by International Playhouse Pictures, it looks as if filmed in a doll house. Everybody's a liar, everything is dream-like, toy-like for no good reason. I'm not saying everything in all movies should be totally realistic, but such unbelievable fantasy things and situations in one movie are way too much. How did they get these fine actors -actresses particularly- to this movie? It's nice to see Mia again; if we were meant to understand why her husband wants to kill her, Mia does do it well. Not funny, not moving, just fake. Stephen Dorff briefly appears at the end, fitting for a play maybe, less for a movie, but this isn't one to measure things at. Terrible."},
        {"role": "system", "content": "negative"},
        {"role": "user", "content": review},
    ]
    try:
        outputs = model.chat(messages=messages, sampling_params=sampling_params)
        text = outputs[0].outputs[0].text
        label = label_map[json.loads(text)["sentiment"]]
        pred_labels.append(label)
        true_labels.append(true_label)
    except Exception as e:
        print(f"Ошибка на примере: {e}")


100%|██████████| 500/500 [10:00<00:00,  1.20s/it]


In [12]:
accuracy_4 = accuracy_score(true_labels, pred_labels)
print(f"Accuracy: {accuracy_4 :.4f}")

Accuracy: 0.8760


In [14]:
columns = ["", "Qwen", "Llama"]
index_labels = ["Zero-Shot", "Few-Shot"]

data = [
    [accuracy_1, accuracy_3],
    [accuracy_2, accuracy_4]

]

df = pd.DataFrame(data, columns=columns[1:], index=index_labels)
print(tabulate(df, headers="keys", tablefmt="github"))


|           |   Qwen |   Llama |
|-----------|--------|---------|
| Zero-Shot |  0.924 |   0.912 |
| Few-Shot  |  0.94  |   0.876 |


**Вывод:**

В режиме Zero-Shot Qwen показывает лучший результат чем Llama, и если ввести несколько примеров, Qwen увеличивает точность, для Llama модель как будто перегружается и точность падает. Qwen лучше адаптируется к примерам. Llama хуже адаптируется к примерам либо плохо настроена, либо хуже работает чем Qwen. В примере на семинаре точность на валидации была значительно ниже.